<center><h1>Xie_Anna_HW4</h1></center>
<br>
<br>

Name: Anna Xie
<br>
Github Username: annajxiee
<br>
USC ID: 2277255370

## 1. Time Series Classification Part 1: Feature Creation/Extraction (HW3 Rerun)

### (a) Obtain Data

Import packages

In [1]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt

Get the AReM Data Set

In [3]:
# (a) Get the AReM Data Set
data_root = '../data'
activities = ['bending1', 'bending2', 'cycling', 'lying', 'sitting', 'standing', 'walking']
cols = ['time', 'avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']

# descrition: reads one file and returns it as a cleaned table
# input: data path (like one to each dataset.csv)
# output: a cleaned pd.df
def load_instance(path):
    df = pd.read_csv(path, comment='#', header=None, names=cols,
                     sep=r'[,\s]+', engine='python', usecols=range(7))
    df = df.drop(columns='time')
    assert df.shape[1] == 6, f'{path}: expected 6 columns, got {df.shape[1]}'
    # helps me check if the amt. of columsn isn't 6
    if df.shape[0] != 480:
        print(f'note: {path} has {df.shape[0]} rows instead of 480')
    return df

instances = []   # (activity, file_num, dataframe)
# for every activity that was measured, flip through the folder for every dataset.csv
for activity in activities:
    folder = os.path.join(data_root, activity)
    # build the folder path and .join inserts the right slash to specify root path
    for f in os.listdir(folder):
        # for every filename in folder, 
        # make sure the if skips anything that isn't a CSV
        if f.endswith('.csv'):
            file_num = int(f.replace('dataset', '').replace('.csv', ''))
            # pulls the number out of the filename and the int turns it into a number so we can keep track of which instance the data belongs to
            instances.append((activity, file_num, load_instance(os.path.join(folder, f))))
            # loads the file and stores a tuple of the label, the file number, and the associated data frame 

print(len(instances))

note: ../data\sitting\dataset8.csv has 479 rows instead of 480
88


### (b) Splitting Data

In [4]:
# (b) Test and Train Data
training, testing = [], []

# for every activity instance in the dataset, unpack each item into three sections
# make sure the df can be split based on criteria
# dataset1+2.csv from bending1+2 and datasets1+2+3.csv => test
# everything else => train
for activity, file_num, df in instances:
    # bending1+2 should specifically be dataset 1 and 2, but otherwise its dataset1-3
    n_test = 2 if activity in ('bending1', 'bending2') else 3
    (testing if file_num <= n_test else training).append((activity, file_num, df))

print(f'{len(training)} train instances, {len(testing)} test instances')   # 69 and 19

69 train instances, 19 test instances


### (c) Feature Extraction

In [ ]:
# extract the min, max, mean, median, sd, Q1, and Q3 from each instance's 6 cols 

# for each csv file, summarize each col into 7 numbers
# new structure: every row is one csv file, every column is one stat number 

# first get everything out of the dataframe
# description: extracts each column grouped by instance number and creates a dictionary of domain features
# input: dataframe
# output: features dictionary
def extract(df, l=1):
  feats = {}
    for i, col in enumerate(df.columns, start=1):                 
        pieces = np.array_split(df[col].to_numpy(), l)            
        for j, seg in enumerate(pieces, start=1):                 
            tag = f'{i}' if l == 1 else f'{i}_{j}'                
            feats[f'min{tag}'] = seg.min()
            feats[f'max{tag}'] = seg.max()
            feats[f'mean{tag}'] = seg.mean()
            feats[f'median{tag}'] = np.median(seg)
            feats[f'std{tag}'] = seg.std(ddof=1)                  
            feats[f'1st_quart{tag}'] = np.percentile(seg, 25)
            feats[f'3rd_quart{tag}'] = np.percentile(seg, 75)
    return feats

# description: builds data frame of domain-features relative to the instance (1 out of 88)
# input: instances (the og training/testing df)
# output: a new df w extracted time-domain features per instance 
def build_feats_table(instances):
  rows = []
  for activity, file_num, df in instances:
    row = {'activity': activity, 'file_num': file_num, 'bending': int(activity in ('bending1', 'bending2'))}
    row.update(extract(df))
    rows.append(row)

  return pd.DataFrame(rows)

training_new = build_feats_table(training)
testing_new = build_feats_table(testing)

all_features = pd.concat([training_new, testing_new], ignore_index=True)
all_features.insert(0, 'Instance', range(1, len(all_features) + 1))

# print(training_new.shape, testing_new.shape, all_features.shape)   # (69, 44) (19, 44) (88, 44)
# all_features.head()
# print(all_features.shape)
# print(all_features)

# standard deviation part too
# estimate the standard deviation of each of the time-domain features you extracted from the data 
# use python boostrapped to build a 90% CI for sd of each feature 

# isolate the feature names themselves
# from the finished df, get only the feat names so not activity, Instance, or file_num
feat_cols = [c for c in all_features.columns if c not in ('Instance', 'activity', 'file_num')]
# print(feat_cols)

X = all_features[feat_cols].values
# print(type(X))
r, c = X.shape
# r = rows, c = cols

nan_counts = all_features[feat_cols].isna().sum()
print(nan_counts[nan_counts > 0])          # which features contain NaN, and how many

rows = all_features[feat_cols].isna().any(axis=1)
print(all_features.loc[rows, ['Instance', 'activity', 'file_num']])   # which instances 
# point estimate: sample stf of each feat
point_std = X.std(axis=0, ddof=1)

# bootstrap: resample 88 instances w replacement,
# recompute every features's standard deviation,
# repeat a certain number of times 
#spread of sd will show how uncertain the estimate iss 
num = np.random.default_rng(42)
times = 2000
boot_stds = np.empty((times, c))

boot_stds = np.array([
    X[num.integers(0, r, size=r)].std(axis=0, ddof=1)
    for _ in range(times)
])

print(boot_stds.shape, np.isnan(boot_stds).sum())
# print(boot_stds)
print(point_std)

# 90% percentile interval: 5% and 95% 
lower = np.percentile(boot_stds, 5, axis=0)
upper = np.percentile(boot_stds, 95, axis=0)

ci_table = pd.DataFrame({
  'std': point_std,
  'ci_lower': lower,
  'ci_upper': upper
}, index=feat_cols)

ci_table.round(4)
# print(type(point_std), type(lower), type(upper))
# print(np.isnan(boot_stds).sum())    # should be 0
# print(boot_stds.shape)              # should be (2000, 42)
# print(np.isnan(X).sum(), X.shape, X.dtype)              # expect 0, (88, 42), float64
# print((np.isnan(boot_stds).sum(axis=1) > 0).sum())     # how many bootstrap rows contain NaN
# print(np.isnan(boot_stds).sum(axis=0))                 # NaN count per feature
print("Three most important things I said were important: mean, standard deviation amd max")

Series([], dtype: int64)
Empty DataFrame
Columns: [Instance, activity, file_num]
Index: []
(2000, 42) 0
[9.56997528 4.39436168 5.33571766 5.44005414 1.7721534  6.15358972
 5.13892461 0.         5.06272937 1.57416392 1.41224416 0.88410548
 0.94638619 2.12526641 2.95646206 4.87513669 4.00838013 4.03639632
 0.94671026 4.22065788 4.17162822 0.         2.18362548 1.16611409
 1.14558566 0.4582417  0.84361968 1.55250418 6.12400143 5.7412383
 5.6755932  5.81378244 1.02489791 6.0964652  5.53172022 0.04583815
 2.51892098 1.15481192 1.08647433 0.51761747 0.75858382 1.52359913]
Three most important things I said were important: mean, standard deviation amd max


## 2. Time Series Classification Part 2: Binary and Multiclass Classification

### (a) Binary Classification Using Logistic Regression

#### i. Plots

In [ ]:
# assume you want to use the training set to classify bending from other activities (classify bending/not bending)
# depict scatter plots of mean, sd, and max from time series 1, 2, ad 6 of each instance
# use color to distinguish bending vs other activities

#### ii. Splitted Plots

#### iii. Time Series Segments

#### iv. Confusion Matrix

#### v. Test Classifier

#### vi. Separation

#### vii. Imbalance

### (b) Binary Classification Using L1-penalized logistic regression

#### i. Time Series Segments

#### ii. Comparison

### (c) Multi-class Classification (The Realistic Case)

#### i. Time Series Segments

#### ii. Naive Bayes

#### ii. Comparison

## 3. ISLR 4.8.3

## 4. ISLR 4.8.7

## 5.1. Extra Practice ISLR 4.8.4

## 5.2. Extra Practice ISLR 4.8.9